# ECQL: дообучение Qwen3-4B-Instruct (QLoRA) — запуск в Google Colab

Ноутбук обучает два LoRA-адаптера (r=8 и r=16), строит график loss, проверяет оба на тестовой
выборке и загружает адаптеры на Hugging Face Hub.

**Как запустить**
1. *Среда выполнения → Сменить среду выполнения → **T4 GPU***.
2. Слева значок 🔑 **Секреты** → добавить `HF_TOKEN` (токен Hugging Face с правом *Write*) и включить доступ для ноутбука.
3. *Среда выполнения → **Выполнить все***. Займёт ~30–40 минут.
4. В конце скачается `llm7_results.zip`; затем *Файл → Скачать → .ipynb*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%capture
# torch уже есть в Colab; остальные версии — те, на которых код проверен
!pip install -q transformers==5.22.0 peft==0.21.2 trl==1.16.0 bitsandbytes==0.50.2 accelerate==1.15.0 datasets==5.1.0

In [ ]:
import os, sys, json, time
REPO = "https://github.com/AlexIntellySoft/llm-7.git"
if not os.path.exists("/content/llm-7"):
    !git clone -q {REPO} /content/llm-7
os.chdir("/content/llm-7")
sys.path.insert(0, "/content/llm-7")
os.makedirs("outputs", exist_ok=True)

from google.colab import userdata
from huggingface_hub import login, whoami
login(token=userdata.get("HF_TOKEN"))
HF_USER = whoami()["name"]
print("Hugging Face:", HF_USER)

## 1. Датасет

In [ ]:
import pandas as pd
from collections import Counter
from src.finetune import load_jsonl
train = load_jsonl("data/train.jsonl")
test = load_jsonl("data/test.jsonl")
print(f"train {len(train)}, test {len(test)}")
print("test по уровням:", dict(Counter(r["level"] for r in test)))
pd.DataFrame(train[:5])[["input", "output"]]

## 2–3. LoRA и обучение

4-bit NF4 (bitsandbytes) + LoRA на `q_proj, k_proj, v_proj, o_proj`, lr 2e-4, 3 эпохи,
батч 2 × накопление 4. Два прогона, отличающихся только рангом: r=8 (alpha=16) и r=16 (alpha=32);
alpha = 2r, чтобы масштаб обновления alpha/r был одинаковым и сравнивался именно ранг.
Loss на тестовой выборке считается только для наблюдения за переобучением — в обучении она не участвует.

In [ ]:
from src.finetune import train_adapter
RUNS = {"r8": (8, 16), "r16": (16, 32)}
histories = {}
for name, (r, alpha) in RUNS.items():
    t0 = time.time()
    histories[name] = train_adapter(train, test, r=r, lora_alpha=alpha, out_dir=f"outputs/adapter_{name}")
    print(f"{name}: {(time.time() - t0) / 60:.1f} мин")

In [ ]:
import matplotlib.pyplot as plt
from src.finetune import loss_points
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (name, hist) in zip(axes, histories.items()):
    ts, tl, es, el = loss_points(hist)
    ax.plot(ts, tl, label="train loss")
    ax.plot(es, el, "o-", label="eval loss (test)")
    ax.set_title(f"LoRA {name}"); ax.set_xlabel("шаг"); ax.grid(alpha=.3); ax.legend()
axes[0].set_ylabel("loss")
plt.tight_layout(); plt.savefig("outputs/loss_curve.png", dpi=150); plt.show()

## 4. Тестирование: адаптеры подключаются к свежей базовой модели

In [ ]:
from src.finetune import load_base, attach_adapters, generate
from src.evaluate import score_one, summarize, METRIC_NAMES
base, tok = load_base()
model = attach_adapters(base, {name: f"outputs/adapter_{name}" for name in RUNS})
metrics, results = {}, {}
for name in RUNS:
    model.set_adapter(name)
    t0 = time.time()
    preds = generate(model, tok, test)
    rows = [{"input": t["input"], "reference": t["output"], "prediction": p, "level": t["level"],
             **score_one(t["output"], p)} for t, p in zip(test, preds)]
    results[name], metrics[name] = rows, summarize(rows)
    with open(f"outputs/predictions_{name}.jsonl", "w") as f:
        f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in rows)
    print(f"{name}: {time.time() - t0:.0f} с")
json.dump(metrics, open("outputs/metrics.json", "w"), ensure_ascii=False, indent=1)
pd.DataFrame(metrics).rename(index=METRIC_NAMES)

In [ ]:
pd.set_option("display.max_colwidth", None)
for name in RUNS:
    bad = [r for r in results[name] if not r["logic_correct"]]
    print(f"\n{name}: ошибок {len(bad)} из {len(test)}")
    for r in bad:
        print(f"  {r['input']}\n    эталон: {r['reference']}\n    модель: {r['prediction']}\n    → {r['error']}")

In [ ]:
pd.DataFrame(results["r16"][:5])[["input", "reference", "prediction", "logic_correct"]]

## 5. Загрузка адаптеров на Hugging Face Hub и выгрузка результатов

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
for name in RUNS:
    repo_id = f"{HF_USER}/qwen3-4b-ecql-lora-{name}"
    api.create_repo(repo_id, exist_ok=True)
    api.upload_folder(folder_path=f"outputs/adapter_{name}", repo_id=repo_id)
    print("https://huggingface.co/" + repo_id)

In [ ]:
!cd outputs && zip -qr ../llm7_results.zip . -x "adapter_*/checkpoint-*"
from google.colab import files
files.download("llm7_results.zip")